In [1]:
import pandas as pd

nfl_clean = pd.read_csv("Data/nfl_data_clean.csv", keep_default_na=False, na_values=[""], low_memory=False)
nfl_clean['Date'] = pd.to_datetime(nfl_clean['Date'])
print(nfl_clean.shape)

(380760, 102)


In [2]:
category_map = {
    'Pass': 'Scrimmage', 'Run': 'Scrimmage', 'Sack': 'Scrimmage',
    'QB Kneel': 'Scrimmage', 'Spike': 'Scrimmage',
    'Kickoff': 'Special teams', 'Punt': 'Special teams',
    'Extra Point': 'Special teams', 'Field Goal': 'Special teams',
    'No Play': 'Penalty (no play)',
}

play_types = sorted(nfl_clean['PlayType'].unique())
dim_playtype = pd.DataFrame({
    'playtype_key': range(1, len(play_types) + 1),
    'play_type': play_types,
})
dim_playtype['play_category'] = dim_playtype['play_type'].map(category_map).fillna('Other')
dim_playtype

,playtype_key,play_type,play_category
0,1,Extra Point,Special teams
1,2,Field Goal,Special teams
2,3,Kickoff,Special teams
3,4,No Play,Penalty (no play)
4,5,Pass,Scrimmage
5,6,Punt,Special teams
6,7,QB Kneel,Scrimmage
7,8,Run,Scrimmage
8,9,Sack,Scrimmage
9,10,Spike,Scrimmage


In [3]:
dates = pd.date_range(nfl_clean['Date'].min(), nfl_clean['Date'].max(), freq='D')
dim_date = pd.DataFrame({'date': dates})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['day_name'] = dim_date['date'].dt.day_name()
# January/February games belong to the previous season
dim_date['nfl_season'] = dim_date['year'].where(dim_date['month'] >= 3, dim_date['year'] - 1)
dim_date = dim_date[['date_key', 'date', 'year', 'month', 'month_name', 'day_name', 'nfl_season']]
print(dim_date.shape)
dim_date.head()

(3035, 7)


,date_key,date,year,month,month_name,day_name,nfl_season
0,20090910,2009-09-10,2009,9,September,Thursday,2009
1,20090911,2009-09-11,2009,9,September,Friday,2009
2,20090912,2009-09-12,2009,9,September,Saturday,2009
3,20090913,2009-09-13,2009,9,September,Sunday,2009
4,20090914,2009-09-14,2009,9,September,Monday,2009


In [4]:
print("playtype_key unique:", dim_playtype['playtype_key'].is_unique)
print("date_key unique:", dim_date['date_key'].is_unique)
print("Play types missing from dim:", set(nfl_clean['PlayType']) - set(dim_playtype['play_type']))
print("All fact dates found in dim_date:", nfl_clean['Date'].isin(dim_date['date']).all())

check = (nfl_clean[['Date', 'Season']].drop_duplicates()
         .merge(dim_date[['date', 'nfl_season']], left_on='Date', right_on='date'))
print("Season mismatches:", (check['Season'] != check['nfl_season']).sum())

playtype_key unique: True
date_key unique: True
Play types missing from dim: set()
All fact dates found in dim_date: True
Season mismatches: 0


In [5]:
team_cols = ['posteam', 'DefensiveTeam', 'HomeTeam', 'AwayTeam']

teams = sorted(pd.concat([nfl_clean[c] for c in team_cols]).dropna().unique())
print(len(teams))
print(teams)

35
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAC', 'JAX', 'KC', 'LA', 'LAC', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'OAK', 'PHI', 'PIT', 'SD', 'SEA', 'SF', 'STL', 'TB', 'TEN', 'WAS']


In [6]:
dim_team = pd.DataFrame({
    'team_key': range(1, len(teams) + 1),
    'team_code': teams,
})
dim_team

,team_key,team_code
0,1,ARI
1,2,ATL
2,3,BAL
3,4,BUF
4,5,CAR
5,6,CHI
6,7,CIN
7,8,CLE
8,9,DAL
9,10,DEN


In [3]:
import pandas as pd

nfl_clean = pd.read_csv("Data/nfl_data_clean.csv", keep_default_na=False, na_values=[""], low_memory=False)
nfl_clean['Date'] = pd.to_datetime(nfl_clean['Date'])
print(nfl_clean.shape)

(380760, 102)


In [4]:
team_cols = ['posteam', 'DefensiveTeam', 'HomeTeam', 'AwayTeam']

teams = sorted(pd.concat([nfl_clean[c] for c in team_cols]).dropna().unique())
print(len(teams))
print(teams)

35
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAC', 'JAX', 'KC', 'LA', 'LAC', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'OAK', 'PHI', 'PIT', 'SD', 'SEA', 'SF', 'STL', 'TB', 'TEN', 'WAS']


In [5]:
dim_team = pd.DataFrame({
    'team_key': range(1, len(teams) + 1),
    'team_code': teams,
})
dim_team

,team_key,team_code
0,1,ARI
1,2,ATL
2,3,BAL
3,4,BUF
4,5,CAR
5,6,CHI
6,7,CIN
7,8,CLE
8,9,DAL
9,10,DEN


In [6]:
for c in team_cols:
    missing = set(nfl_clean[c].dropna()) - set(dim_team['team_code'])
    print(c, "| nulls:", nfl_clean[c].isnull().sum(), "| codes missing from dim:", missing)

print("team_key unique:", dim_team['team_key'].is_unique)
print("team_code unique:", dim_team['team_code'].is_unique)

posteam | nulls: 0 | codes missing from dim: set()
DefensiveTeam | nulls: 0 | codes missing from dim: set()
HomeTeam | nulls: 0 | codes missing from dim: set()
AwayTeam | nulls: 0 | codes missing from dim: set()
team_key unique: True
team_code unique: True


In [7]:
for c in ['JAC', 'JAX', 'STL', 'LA', 'SD', 'LAC']:
    mask = (nfl_clean['posteam'] == c) | (nfl_clean['DefensiveTeam'] == c)
    print(c, sorted(nfl_clean.loc[mask, 'Season'].unique()))

JAC [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]
JAX [np.int64(2016), np.int64(2017)]
STL [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]
LA [np.int64(2016), np.int64(2017)]
SD [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]
LAC [np.int64(2017)]


In [8]:
franchise_map = {'JAC': 'JAX', 'STL': 'LA', 'SD': 'LAC'}
dim_team['franchise_code'] = dim_team['team_code'].replace(franchise_map)
print(dim_team['franchise_code'].nunique())

32


In [10]:
category_map = {
    'Pass': 'Scrimmage', 'Run': 'Scrimmage', 'Sack': 'Scrimmage',
    'QB Kneel': 'Scrimmage', 'Spike': 'Scrimmage',
    'Kickoff': 'Special teams', 'Punt': 'Special teams',
    'Extra Point': 'Special teams', 'Field Goal': 'Special teams',
    'No Play': 'Penalty (no play)',
}

play_types = sorted(nfl_clean['PlayType'].unique())
dim_playtype = pd.DataFrame({
    'playtype_key': range(1, len(play_types) + 1),
    'play_type': play_types,
})
dim_playtype['play_type'] = dim_playtype['play_type']
dim_playtype['play_category'] = dim_playtype['play_type'].map(category_map).fillna('Other')

print("dim_team defined:", 'dim_team' in globals())
print("dim_date defined:", 'dim_date' in globals())

dim_team defined: True
dim_date defined: False


In [11]:
fact_plays = nfl_clean.copy()

fact_plays['date_key'] = fact_plays['Date'].dt.strftime('%Y%m%d').astype(int)

pt_map = dict(zip(dim_playtype['play_type'], dim_playtype['playtype_key']))
fact_plays['playtype_key'] = fact_plays['PlayType'].map(pt_map)

team_map = dict(zip(dim_team['team_code'], dim_team['team_key']))
fact_plays['posteam_key'] = fact_plays['posteam'].map(team_map)
fact_plays['defteam_key'] = fact_plays['DefensiveTeam'].map(team_map)
fact_plays['hometeam_key'] = fact_plays['HomeTeam'].map(team_map)
fact_plays['awayteam_key'] = fact_plays['AwayTeam'].map(team_map)

In [12]:
dates = pd.date_range(nfl_clean['Date'].min(), nfl_clean['Date'].max(), freq='D')
dim_date = pd.DataFrame({'date': dates})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['day_name'] = dim_date['date'].dt.day_name()
dim_date['nfl_season'] = dim_date['year'].where(dim_date['month'] >= 3, dim_date['year'] - 1)
dim_date = dim_date[['date_key', 'date', 'year', 'month', 'month_name', 'day_name', 'nfl_season']]
print(dim_date.shape)

(3035, 7)


In [13]:
fact_cols = [
    'GameID', 'Drive', 'qtr', 'down', 'ydstogo', 'yrdline100',
    'date_key', 'playtype_key', 'posteam_key', 'defteam_key',
    'hometeam_key', 'awayteam_key',
    'PosTeamScore', 'DefTeamScore', 'ScoreDiff',
    'Yards.Gained', 'FirstDown', 'Touchdown',
    'AirYards', 'YardsAfterCatch', 'Penalty.Yards',
    'EPA', 'WPA', 'Win_Prob',
    'Passer', 'Receiver', 'Rusher',
]
fact_plays = fact_plays[fact_cols].reset_index(drop=True)
fact_plays.insert(0, 'play_id', range(1, len(fact_plays) + 1))
print(fact_plays.shape)

(380760, 28)


In [14]:
key_cols = ['date_key', 'playtype_key', 'posteam_key', 'defteam_key', 'hometeam_key', 'awayteam_key']
print("Null keys:\n", fact_plays[key_cols].isnull().sum())
print("play_id unique:", fact_plays['play_id'].is_unique)
print("Rows match source:", len(fact_plays) == len(nfl_clean))
print("All fact dates in dim_date:", fact_plays['date_key'].isin(dim_date['date_key']).all())

Null keys:
 date_key        0
playtype_key    0
posteam_key     0
defteam_key     0
hometeam_key    0
awayteam_key    0
dtype: int64
play_id unique: True
Rows match source: True
All fact dates in dim_date: True


In [15]:
dim_team.to_csv("Data/dim_team.csv", index=False)
dim_playtype.to_csv("Data/dim_playtype.csv", index=False)
dim_date.to_csv("Data/dim_date.csv", index=False)
fact_plays.to_csv("Data/fact_plays.csv", index=False)

for name, df in [('dim_team', dim_team), ('dim_playtype', dim_playtype),
                 ('dim_date', dim_date), ('fact_plays', fact_plays)]:
    print(name, df.shape)

dim_team (35, 3)
dim_playtype (10, 3)
dim_date (3035, 7)
fact_plays (380760, 28)


In [16]:
dim_team.to_csv("Data/dim_team.csv", index=False)
dim_playtype.to_csv("Data/dim_playtype.csv", index=False)
dim_date.to_csv("Data/dim_date.csv", index=False)
fact_plays.to_csv("Data/fact_plays.csv", index=False)

for name, df in [('dim_team', dim_team), ('dim_playtype', dim_playtype),
                 ('dim_date', dim_date), ('fact_plays', fact_plays)]:
    print(name, df.shape)

dim_team (35, 3)
dim_playtype (10, 3)
dim_date (3035, 7)
fact_plays (380760, 28)
